In [ ]:
import numpy as np
import pandas as pd
from scripts.plotting import *
from scripts.denoising import *
from sklearn.preprocessing import StandardScaler
from scipy.linalg import svd
from scipy.spatial.distance import pdist, squareform, jaccard
from scipy.linalg import norm

In [ ]:
import scanpy as sc
import scvelo as scv

bdata = sc.read_h5ad("./data/pancreas/pancreas_inferred_velocity.h5ad")
scv.pl.velocity_embedding_stream(bdata, basis="umap", color="clusters", density=1.5, arrow_size=0.1)

In [ ]:
sc.pp.neighbors(bdata, use_rep="X_pca")  # Using PCA representation
sc.tl.umap(bdata, min_dist=0.3)
sc.tl.leiden(bdata, key_added="clusters_expression")
sc.pl.umap(bdata, color=["clusters_expression"])

In [ ]:
# Extract velocity matrix
velocity_matrix = bdata.layers["velocity"]

# Perform PCA on velocity data
pca = PCA(n_components=30)
velocity_PCA = pca.fit_transform(velocity_matrix)

# Store PCA result in obsm
bdata.obsm["velocity_PCA"] = velocity_PCA

# Compute neighbors using velocity PCA
sc.pp.neighbors(bdata, use_rep="velocity_PCA")

# Cluster based on velocity PCA
sc.tl.leiden(bdata, key_added="clusters_velocity")

sc.pp.neighbors(bdata, use_rep="X_pca")  # Using PCA representation
sc.tl.umap(bdata, min_dist=0.3)
# Visualize clustering results
sc.pl.umap(bdata, color=["clusters_expression", "clusters_velocity"])

In [ ]:
# Extract velocity matrix
velocity_matrix = bdata.layers["velocity"]

# Perform PCA on velocity data
pca = PCA(n_components=30)
velocity_PCA = pca.fit_transform(velocity_matrix)

# Store PCA result in obsm
bdata.obsm["velocity_PCA"] = velocity_PCA

# Compute neighbors using velocity PCA
sc.pp.neighbors(bdata, use_rep="velocity_PCA")

# Cluster based on velocity PCA
sc.tl.leiden(bdata, key_added="clusters_velocity")

sc.tl.umap(bdata, min_dist=0.3)
# Visualize clustering results
sc.pl.umap(bdata, color=["clusters_expression", "clusters_velocity"])

In [ ]:
# Perform PCA on unspliced matrix
unspliced_matrix = bdata.layers["Ms"]

pca = PCA(n_components=30)
unspliced_PCA = pca.fit_transform(unspliced_matrix)

# Store PCA result in obsm
bdata.obsm["unspliced_PCA"] = unspliced_PCA

# Compute neighbors using unspliced PCA
sc.pp.neighbors(bdata, use_rep="unspliced_PCA")

# Cluster based on unspliced PCA
sc.tl.leiden(bdata, key_added="clusters_unspliced")

sc.tl.umap(bdata, min_dist=0.3)
# Visualize clustering results
sc.pl.umap(bdata, color=["clusters_expression", "clusters_velocity", "clusters_unspliced"])

In [ ]:
sc.pp.neighbors(bdata, use_rep="X_pca")  # Using PCA representation
sc.tl.umap(bdata, min_dist=0.3)
# Visualize clustering results
sc.pl.umap(bdata, color=["clusters_expression", "clusters_velocity", "clusters_unspliced"])

In [ ]:
sc.pp.neighbors(bdata, use_rep="velocity_PCA")
# Cluster based on velocity PCA
sc.tl.leiden(bdata, key_added="clusters_velocity")

sc.tl.umap(bdata, min_dist=0.3)
# Visualize clustering results
sc.pl.umap(bdata, color=["clusters_expression", "clusters_velocity", "clusters_unspliced"])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Randomly select 25 genes
np.random.seed(42)  # For reproducibility
random_genes = np.random.choice(bdata.var_names, size=25, replace=False)

# Plot scatterplots in a 5x5 grid
fig, axes = plt.subplots(5, 5, figsize=(20, 20))

for idx, gene in enumerate(random_genes):
    row, col = divmod(idx, 5)  # Convert index to 2D grid position
    spliced = bdata.layers["Ms"][:, bdata.var_names.get_loc(gene)]
    unspliced = bdata.layers["Mu"][:, bdata.var_names.get_loc(gene)]
    
    axes[row, col].scatter(spliced, unspliced, alpha=0.5, s=5)
    axes[row, col].set_xlabel("Spliced")
    axes[row, col].set_ylabel("Unspliced")
    axes[row, col].set_title(f"Gene: {gene}")

plt.tight_layout()
plt.show()


In [ ]:
from scipy.spatial.distance import cdist

def brute_force_min_distance(X, V, epsilon=0.5, step=0.1):
    """
    1) For t1, t2 in {0, 0.1, ..., epsilon}, compute 
         M_{(t1,t2)} = pairwise_distance_matrix( X + t1*V, X + t2*V ).
    2) Collect all M_{(t1,t2)} in a 6x6 list (since 0..0.5 in steps of 0.1 => 6 points).
    3) Return also the element-wise minimum across all 6x6 = 36 matrices.

    X: shape (n, d)
    V: shape (n, d)
    epsilon: float
    step: float
    Returns:
       - all_matrices: a 6x6 list-of-lists of (n x n) arrays
       - min_matrix: the (n x n) array of element-wise minima across all 36 matrices.
    """
    t_values = np.arange(-epsilon, epsilon + 1e-9, step)  # e.g. [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
    n, d = X.shape

    # Store all (n x n) matrices in a 3D array
    all_matrices = []
    
    for t1 in t_values:
        print("------")
        print(t1)
        for t2 in t_values:
            print(t2)
            A = X + t1 * V  # shape (n, d)
            B = X + t2 * V  # shape (n, d)
            dist_mat = cdist(A, B, metric='euclidean')  # shape (n, n)
            all_matrices.append(dist_mat)

    # Convert to a (n x n x 36) array
    all_matrices = np.stack(all_matrices, axis=-1)  # shape (n, n, 36)

    # Compute the element-wise minimum across the 3rd axis
    min_matrix = np.min(all_matrices, axis=-1)  # shape (n, n)

    return all_matrices, min_matrix

In [ ]:
bdata

In [ ]:
X = bdata.layers["Ms"]
V = bdata.layers["velocity"]
dist_matrix = brute_force_min_distance(X, V)